# 方案一：凸包热力学初筛


## 1. 参数配置


In [ ]:
# ============================================================
# User Config Area — modify here
# ============================================================

# --- chemical system ---
system = ["Fe", "Si", "S"]

# --- near-stable threshold (eV/atom) ---
HULL_THRESHOLD = 0.1

# --- MACE-MP-0 enumeration ---
MAX_ATOMS = 8               # max atoms per formula unit
VOL_PER_ATOM = 18.0         # initial vol per atom (Ang^3)
M3G_FMAX = 0.05             # relaxation force convergence (eV/Ang)

# --- debug mode ---
DEBUG_MODE = True          # True = 只计算少量缺失结构，快速调试
DEBUG_N_MISSING = 3          # 调试模式下计算的缺失结构数量


## 2. 导入库与 API 设置


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fix OpenMP conflict
from dotenv import load_dotenv
from mp_api.client import MPRester
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter, PDEntry
from pymatgen.entries.mixing_scheme import MaterialsProjectDFTMixingScheme
import matplotlib.pyplot as plt

%matplotlib inline

# MACE-MP-0
try:
    from mace.calculators import mace_mp
    from ase.optimize import BFGS
    from ase import Atoms
    import torch
    _HAS_MACE = True
except ImportError:
    _HAS_MACE = False

env_loaded = load_dotenv(os.path.join("..", "api", "myapi.env"))
if not env_loaded:
    print("Warning: myapi.env not found")
    print(f"CWD: {os.getcwd()}")

API_KEY = os.getenv("MP_API_KEY")
if not API_KEY:
    raise ValueError("API Key missing")
else:
    print(f"\nAPI Key loaded (prefix: {API_KEY[:8]}...)")
    print("Ready.")


## 3. 获取 MP 数据与构建凸包


In [ ]:
import time

MAX_DB_RETRIES = 3
for attempt in range(1, MAX_DB_RETRIES + 1):
    try:
        with MPRester(API_KEY) as mpr:
            # 1. 获取体系数据
            entries = mpr.get_entries_in_chemsys(system, include_structure=True,
                                            additional_criteria={"thermo_types": ["GGA_GGA+U"]})
            print(f"获取到 {len(entries)} 条计算条目")

            # 2. 能量修正 + 构建凸包
            scheme = MaterialsProjectDFTMixingScheme()
            entries = scheme.process_entries(entries)
            pd = PhaseDiagram(entries)
            print(f"凸包相图构建完成 (0K, {len(entries)} 条)")
            print("   → 相图在下一步 Cell 中统一绘制（含 MACE-MP-0 对比）")
        break  # 成功则跳出重试循环
    except Exception as e:
        print(f"⚠️ 第 {attempt}/{MAX_DB_RETRIES} 次连接失败: {e}")
        if attempt < MAX_DB_RETRIES:
            wait_sec = 2 ** attempt  # 指数退避: 2s, 4s, 8s
            print(f"   {wait_sec} 秒后重试...")
            time.sleep(wait_sec)
        else:
            print(f"❌ 已重试 {MAX_DB_RETRIES} 次仍失败，请检查网络连接和 API Key")
            raise


## 4. MACE-MP-0 缺失结构能量预测


In [ ]:
# ============================================================
# MACE-MP-0 缺失结构能量预测
# 对 MP 数据库中不存在的候选结构，使用 MACE-MP-0 通用力场
# 弛豫并预测能量，纳入凸包热力学分析
#
# 实际项目流程：自建前驱体反应库 → 枚举候选物相 →
#   MP 已有 → 直接采用  |  MP 缺失 → MACE-MP-0 预测 → 合并凸包分析
# ============================================================

if not _HAS_MACE:
    print("MACE-MP-0 未安装，跳过。pip install mace-torch ase")
else:
    from pymatgen.entries.computed_entries import ComputedStructureEntry
    from pymatgen.core import Composition, Structure, Lattice
    import numpy as np
    import warnings
    warnings.filterwarnings("ignore")

    print("=" * 60)
    print("MACE-MP-0 缺失结构能量预测")
    print("=" * 60)

    # ---------- 1. 加载 MACE-MP-0 ----------
    print("\n[1/5] 加载 MACE-MP-0 通用力场...")
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"      设备: {_device}")

    # ★★★ 在这里修改您的模型文件路径 ★★★
    model_path = "2023-12-03-mace-128-L1_epoch-199.model"

    try:
        calculator = mace_mp(model=model_path, device=_device, default_dtype="float64")
        print("      模型加载完成")
        _MACE_OK = True
    except Exception as _e:
        _MACE_OK = False
        print(f"      FAILED: {_e}")
        print("      Skipping MACE prediction.")

    if not _MACE_OK:
        pass
    else:

        # ---------- 2. 快速验证 ----------
        print("\n[2/5] 验证 MACE-MP-0 预测精度（取 1 个 MP 条目对比）...")
        test_entry = [e for e in entries if hasattr(e, "structure") and e.structure.num_sites <= 30][0]
        test_struct = test_entry.structure
        mp_energy_pa = test_entry.energy / test_struct.num_sites

        atoms = test_struct.to_ase_atoms()
        atoms.calc = calculator
        opt = BFGS(atoms, trajectory=None, logfile=None)
        opt.run(fmax=M3G_FMAX, steps=500)
        mace_energy_pa = atoms.get_potential_energy() / test_struct.num_sites

        delta = abs(mace_energy_pa - mp_energy_pa)
        print(f"      验证条目: {test_entry.entry_id} ({test_struct.formula})")
        print(f"      MP 能量:        {mp_energy_pa:.4f} eV/atom")
        print(f"      MACE-MP-0 能量: {mace_energy_pa:.4f} eV/atom")
        print(f"      偏差:           {delta:.4f} eV/atom")

        # ---------- 3. 扫描 MP 化学式 ----------
        print("\n[3/5] 扫描 MP 已覆盖的化学式...")
        mp_formulas = {e.composition.reduced_formula for e in entries}
        print(f"      MP 已覆盖 {len(mp_formulas)} 种化学式")

        # ---------- 4. 识别缺失项 ----------
        print("\n[4/5] 识别 MP 缺失的候选物相...")
        from itertools import combinations_with_replacement
        max_atoms = MAX_ATOMS
        candidate_comps = []
        n_el = len(system)
        for total in range(2, max_atoms + 1):
            def gen_comp(n_remain, dim, prefix):
                if dim == 1:
                    yield prefix + (n_remain,)
                else:
                    for x in range(1, n_remain - (dim - 1) + 1):
                        yield from gen_comp(n_remain - x, dim - 1, prefix + (x,))
            for vec in gen_comp(total, n_el, ()):
                el_map = {system[i]: v for i, v in enumerate(vec)}
                candidate_comps.append(Composition(el_map))
        seen = set()
        candidate_comps_dedup = []
        for c in candidate_comps:
            rf = c.reduced_formula
            if rf not in seen:
                seen.add(rf)
                candidate_comps_dedup.append(c)
        candidate_comps = candidate_comps_dedup
        print(f"      自动生成 {len(candidate_comps)} 个候选化学式")

        missing_comps = []
        for comp in candidate_comps:
            rf = comp.reduced_formula
            if rf in mp_formulas:
                print(f"      [skip] {comp.formula} — MP 已覆盖")
            else:
                print(f"      [MACE] {comp.formula} — 缺失，将由 MACE-MP-0 预测")
                missing_comps.append(comp)


        if DEBUG_MODE:
            missing_comps = missing_comps[:DEBUG_N_MISSING]
            print(f"      [DEBUG] 调试模式：仅处理前 {len(missing_comps)} 个缺失结构")

        # ---------- 5. MACE-MP-0 弛豫 ----------
        print("\n[5/5] MACE-MP-0 弛豫并预测缺失结构能量...")
        mace_entries = []

        for comp in missing_comps:
            print(f"\n      >> {comp.formula} ...")

            num_atoms = int(sum(comp.values()))
            vol_per_atom = VOL_PER_ATOM
            a = (num_atoms * vol_per_atom) ** (1 / 3)
            lattice = Lattice.cubic(a)

            species = []
            for el, amt in comp.items():
                species.extend([el] * int(amt))

            np.random.seed(42)
            coords = np.random.uniform(0.15, 0.85, (num_atoms, 3))
            init_struct = Structure(lattice, species, coords)
            print(f"         初始: {num_atoms} atoms, cubic a={a:.1f} A")

            try:
                atoms = init_struct.to_ase_atoms()
                atoms.calc = calculator
                opt = BFGS(atoms, trajectory=None, logfile=None)
                opt.run(fmax=M3G_FMAX, steps=500)
                final_energy = atoms.get_potential_energy()
                energy_pa = final_energy / num_atoms
                print(f"         弛豫完成: E = {energy_pa:.4f} eV/atom")
                # 能量合理性检查
                if abs(energy_pa) > 100:
                    print(f"         SKIP: energy out of range ({energy_pa:.1f} eV/atom)")
                    continue

                # build structure from relaxed atoms
                from ase.io import write as ase_write
                import tempfile, os as _os
                with tempfile.NamedTemporaryFile(suffix=".cif", delete=False) as tf:
                    ase_write(tf.name, atoms, format="cif")
                    tmp_path = tf.name
                from pymatgen.io.cif import CifParser
                final_struct = CifParser(tmp_path).get_structures()[0]
                _os.unlink(tmp_path)

                entry = ComputedStructureEntry(
                    structure=final_struct,
                    energy=final_energy,
                )
                entry.data["source"] = "MACE-MP-0"
                entry.entry_id = f"MACE-{comp.reduced_formula}"
                mace_entries.append(entry)
            except Exception as ex:
                print(f"         FAILED: {ex}")

        # ---------- 6. 合并凸包 ----------
        if mace_entries:
            n_mp = len(entries)
            n_mace = len(mace_entries)
            print(f"\n{'=' * 60}")
            print(f"合并 MP ({n_mp} 条) + MACE-MP-0 ({n_mace} 条) -> 重建凸包")
            print(f"{'=' * 60}")

            entries_combined = list(entries) + mace_entries
            pd_combined = PhaseDiagram(entries_combined)

            entries_orig = entries
            pd_orig = pd
            entries = entries_combined
            pd = pd_combined

            import plotly.io as pio
            pio.renderers.default = "notebook"
            print()
            print("=" * 50)
            print("MP only (" + str(n_mp) + " entries)")
            print("=" * 50)
            try:
                PDPlotter(pd_orig, show_unstable=True).show()
            except Exception as _pe:
                print(f"      PDPlotter error (mp): {_pe}")
            print()
            print("=" * 50)
            print("MP + MACE-MP-0 (" + str(n_mp+n_mace) + " entries)")
            print("=" * 50)
            try:
                PDPlotter(pd_combined, show_unstable=True).show()
            except Exception as _pe:
                print(f"      PDPlotter error (combined): {_pe}")
            print()

            near_stable = [e for e in entries_combined
                if pd_combined.get_e_above_hull(PDEntry(e.composition, e.energy)) is not None
                and pd_combined.get_e_above_hull(PDEntry(e.composition, e.energy)) < HULL_THRESHOLD]
            if near_stable:
                best = {}
                for e in near_stable:
                    rf = e.composition.reduced_formula
                    h = pd_combined.get_e_above_hull(PDEntry(e.composition, e.energy))
                    if rf not in best or (h is not None and (best[rf][1] is None or h < best[rf][1])):
                        best[rf] = (e, h)
                near_stable_dedup = [v[0] for v in best.values()]
                pd_near = PhaseDiagram(near_stable_dedup)
                print("Near-stable (" + str(len(near_stable_dedup)) + " entries):")
                try:
                    PDPlotter(pd_near, show_unstable=True).show()
                except Exception as _pe:
                    print(f"      PDPlotter error (near): {_pe}")
            else:
                print("no near-stable entries")
        else:
            print("\n所有候选结构均已在 MP 中，无需 MACE-MP-0 预测。")


In [ ]:
# ============================================================
# 导出条目供方案2使用
# 保存当前 entries (含 MP + MACE-MP-0 合并结果) 为 JSON
# 方案2 将优先从此文件加载，避免重复拉取 MP 数据
# ============================================================
import json
import os
from datetime import datetime
from monty.json import MontyEncoder

# 前置检查：确保上游 Cell 已运行
if "entries" not in dir():
    raise RuntimeError("❌ 变量 entries 未定义，请先运行前面的 Cell 5 (获取数据) 和 Cell 7 (MACE-MP-0 预测)！")

# 统计来源
n_mp = sum(1 for e in entries if getattr(e, 'data', {}).get('source', '') != 'MACE-MP-0')
n_mace = sum(1 for e in entries if getattr(e, 'data', {}).get('source', '') == 'MACE-MP-0')

# 安全序列化：三重防护处理 MP 条目 data 中的 Element 对象 key
bad_keys = 0
entry_dicts = []
for e in entries:
    try:
        entry_dicts.append(e.as_dict())
    except TypeError:
        bad_keys += 1
        # 递归将所有 dict key 转字符串
        def _fix_keys(obj):
            if isinstance(obj, dict):
                return {str(k): _fix_keys(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [_fix_keys(v) for v in obj]
            return obj
        e.data = _fix_keys(e.data) if hasattr(e, 'data') and e.data else {}
        try:
            entry_dicts.append(e.as_dict())
        except TypeError:
            # 最后手段：手动构建
            d = {
                "@module": "pymatgen.entries.computed_entries",
                "@class": "ComputedStructureEntry",
                "composition": e.composition.as_dict(),
                "energy": e.energy,
                "entry_id": getattr(e, "entry_id", "unknown"),
                "correction": getattr(e, "correction", 0.0),
                "data": json.loads(json.dumps(_fix_keys(e.data), cls=MontyEncoder)),
                "structure": e.structure.as_dict() if hasattr(e, "structure") else None,
            }
            entry_dicts.append(d)

if bad_keys > 0:
    print(f"⚠️ {bad_keys} 个条目的 data 含非标准 key，已自动修复")

export_data = {
    "schema_version": 1,
    "created_at": datetime.now().isoformat(),
    "system": system,
    "n_entries_mp": n_mp,
    "n_entries_mace": n_mace,
    "n_entries_total": len(entries),
    "entries": entry_dicts,
}

output_path = "scheme1_export.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, cls=MontyEncoder, indent=2)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"✅ 已导出至 {output_path}")
print(f"   体系: {"-".join(system)}")
print(f"   条目: {len(entries)} 条 (MP: {n_mp}, MACE-MP-0: {n_mace})")
print(f"   文件大小: {file_size_mb:.1f} MB")
print(f"   → 方案2 可直接从此文件加载，跳过 MP API 调用")


## 5. 查看晶体结构信息


In [ ]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core import Composition

# 筛选所有化合物（同时含目标体系所有元素的条目）
compounds = [e for e in entries
             if all(e.composition.get(el, 0) > 0 for el in system)]

print(f"共找到 {len(compounds)} 个化合物条目：\n")
for i, e in enumerate(compounds):
    struct = e.structure
    try:
        sga = SpacegroupAnalyzer(struct, symprec=0.1)
        spg_num = sga.get_space_group_number()
        spg_symbol = sga.get_space_group_symbol()
    except Exception:
        spg_num = "?"
        spg_symbol = "无法确定"
    
    print(f"条目 {i}: {getattr(e, 'entry_id', 'N/A')}")
    pdentry = PDEntry(e.composition, e.energy)
    e_hull = pd.get_e_above_hull(pdentry)
    if e_hull is None:
        tag = "N/A"
    elif e_hull < 0.001:
        tag = "🟢 稳定"
    elif e_hull < HULL_THRESHOLD:
        tag = "🟡 近稳"
    else:
        tag = "🟠 亚稳态"
    print(f"  稳定性: {tag} (e_hull={e_hull:.4f})")
    print(f"  化学式: {e.structure.formula}")
    print(f"  空间群: #{spg_num} {spg_symbol}")
    print(f"  能量: {e.energy:.4f} eV/atom")
    print()


## 6. 计算形成能与 e_above_hull


In [ ]:
# ============================================================
# 对所有化合物计算形成能和 e_above_hull，并写入 output.md
# ============================================================

from pymatgen.core import Composition

# 筛选所有化合物
compounds = [e for e in entries
             if all(e.composition.get(el, 0) > 0 for el in system)]

system_name = "-".join(system)

# 收集结果行，同时打印并写入 markdown
md_lines = []
md_lines.append(f"# {system_name} 体系 DFT 凸包热力学初筛结果\n")
md_lines.append(f"\n**条目总数**: {len(entries)} | **化合物数**: {len(compounds)}\n")
md_lines.append("\n| # | 化学式 | entry_id | 形成能 (eV/atom) | e_above_hull (eV/atom) | 稳定性 |\n")
md_lines.append("|---|--------|----------|-------------------|------------------------|--------|\n")

if compounds:
    print(f"共 {len(compounds)} 个化合物：\n")
    for idx, e in enumerate(compounds, 1):
        pdentry = PDEntry(e.composition, e.energy)
        form_e = pd.get_form_energy_per_atom(pdentry)
        e_above = pd.get_e_above_hull(pdentry)
        
        # 稳定性判断
        if e_above is None:
            stability = "N/A"
        elif e_above < 0.001:
            stability = "🟢 稳定"
        elif e_above < HULL_THRESHOLD:
            stability = "🟡 近稳"
        else:
            stability = "🟠 亚稳态"
        
        # 终端打印
        print(f"🧪 {e.composition.formula}  ({getattr(e, 'entry_id', 'N/A')})")
        print(f"   🔥 形成能: {form_e:.4f} eV/atom")
        if e_above is None:
            print(f"   📊 e_above_hull: N/A")
        else:
            print(f"   📊 e_above_hull: {e_above:.4f} eV/atom  {stability}")
        print()
        
        # 追加 markdown 表格行
        e_above_str = f"{e_above:.4f}" if e_above is not None else "N/A"
        md_lines.append(f"| {idx} | {e.composition.formula} | {getattr(e, 'entry_id', 'N/A')} | {form_e:.4f} | {e_above_str} | {stability} |\n")
    
    # 写入 output.md
    with open("output.md", "w", encoding="utf-8") as f:
        f.writelines(md_lines)
    print("📄 结果已导出至 output.md")
else:
    print("❌ 未在下载的数据中找到化合物")
